# Stage1 原始 FT‑900 半导体出口：断点未知版条件性信息检验

冻结原 Stage1 39 项输入，双方共有发行日龄、分类时期和断点未知控制；仅新增同报告出口环比。2012‑01 事件后 23 个训练信号保持数值未知，使用训练样本的中位数填补，不缩短日历。历史首次公开 PDF 字节未证实，结果只能作为开发期反馈。


In [1]:
import json,hashlib
from pathlib import Path
import numpy as np,pandas as pd,qlib,sklearn
from qlib.workflow import R
from qlib.data.dataset import DatasetH
from qlib.data.dataset.handler import DataHandlerLP
from src.data.minute_intake import atomic_json
root=Path.cwd();card=root/'research/stage1_ft900_export_unknown_information_20261023';plan=json.loads((card/'plan.json').read_text())
def sha(path):return hashlib.sha256(Path(path).read_bytes()).hexdigest()
assert sklearn.__version__==plan['model']['runtime_scikit_learn']
amend=json.loads((card/'design_inspection_amendment.json').read_text())
budget_amend=json.loads((card/'pre_fit_budget_amendment.json').read_text())
assert amend['new_plan_sha256']==budget_amend['old_plan_sha256'] and budget_amend['new_plan_sha256']==sha(card/'plan.json')
assert amend['source_Recorder_object_loads_before_preflight']==1 and budget_amend['source_Recorder_loads_max_after_amendment']==16
old=plan['replaces_invalid_unfitted_design'];old_card=root/old['card']
assert sha(old_card/'design_halt.json')==old['design_halt_sha256']
assert sha(old_card/'source_conflict_design_review.json')==old['source_conflict_review_sha256']
assert json.loads((old_card/'summary.json').read_text())['model_fits']==0
spec=plan['candidate_source'];source_card=root/spec['card'];parent=plan['parent'];parent_card=root/parent['source_card']
for key,file in [('summary_sha256','summary.json'),('availability_audit_sha256','availability_audit.json'),('readback_sha256','readback_verification.json'),('independent_review_sha256','independent_review.json'),('repair_1_sha256','mechanical_repair_1.json'),('repair_2_sha256','mechanical_repair_2.json')]:assert sha(source_card/file)==spec[key],('source changed',file)
assert sha(root/'research/census_ft900_index_boundary_20261023/transition_adjudication.json')==spec['transition_adjudication_sha256']
assert json.loads((source_card/'summary.json').read_text())['nas_evidence']['sha256']==spec['nas_archive_sha256']
assert json.loads((source_card/'independent_review.json').read_text())['fit_authorized'] is False
for name,key in [('inputs/features','feature_artifact_sha256'),('inputs/daily_frame','frame_artifact_sha256'),('inputs/timing','timing_artifact_sha256'),('models/base_outer_71','base_model_artifact_sha256')]:assert sha(parent_card/'qlib_recorder'/parent['experiment_id']/parent['recorder_id']/'artifacts'/name)==parent[key]
source_artifact=source_card/'qlib_recorder'/spec['experiment_id']/spec['recorder_id']/'artifacts'/spec['mapped_artifact']
assert sha(source_artifact)==spec['mapped_artifact_sha256']
assert sha(source_card/'qlib_recorder'/spec['experiment_id']/spec['recorder_id']/'artifacts/metadata/dataset')==spec['dataset_artifact_sha256']
qlib.init(provider_uri=str(root/'data/qlib/minute_information_pit_allocator_20261012/daily_prices'),region='us',kernels=1)
with R.uri_context(str(source_card/'qlib_recorder')):
 source_rec=R.get_recorder(recorder_id=spec['recorder_id'],experiment_id=spec['experiment_id'])
 mapped=source_rec.load_object(spec['mapped_artifact'])
with R.uri_context(str(parent_card/'qlib_recorder')):
 rec=R.get_recorder(recorder_id=parent['recorder_id'],experiment_id=parent['experiment_id'])
 original=rec.load_object('models/base_outer_71');base=rec.load_object('inputs/features')
 frame=rec.load_object('inputs/daily_frame');timing=rec.load_object('inputs/timing')
base_names=list(original.base_features);assert len(base_names)==39 and set(base_names).issubset(base.columns)
assert frame.index.equals(base.index) and frame.index.equals(timing.index)
assert len(mapped)==3964 and mapped.signal_date.is_unique and int(mapped.transition_quarantined.sum())==23
aligned=mapped.set_index('signal_date').reindex(frame.index)
name=plan['single_mechanism']['feature'];feature=aligned.mom_pct.astype(float).rename(name)
controls=pd.DataFrame(index=frame.index)
controls['release_age_days']=aligned.release_age_days.astype(float)
controls['definition_post_2012']=aligned.reference_month.ge('2012-01').astype(float)
controls['transition_unknown']=aligned.transition_quarantined.astype(float)
control_names=list(controls.columns)
forward=frame['soxx_forward_open_return'].astype(float)
parts=[forward.shift(-i) for i in range(5)]
valid=np.logical_and.reduce([np.isfinite(x.to_numpy(float))&(x.to_numpy(float)>-1) for x in parts])
label=pd.Series(np.log1p(pd.concat(parts,axis=1)).sum(axis=1).where(valid),index=frame.index,name='soxx_forward_five_open_log_return')
mature=pd.to_datetime(timing.return_available_at.shift(-4),utc=True)
cutoff=pd.Timestamp('2023-01-01',tz='America/New_York').tz_convert('UTC')
last=pd.Timestamp('2026-10-10',tz='America/New_York').tz_convert('UTC')
train_dates=(frame.index>=pd.Timestamp('2011-01-03'))&(frame.index<=pd.Timestamp('2022-12-30'))&(mature<cutoff)&label.notna()
held_dates=(frame.index>=pd.Timestamp('2023-01-03'))&(frame.index<=pd.Timestamp('2026-10-07'))&(mature<last)&label.notna()
assert train_dates.sum()==plan['cohorts']['expected_mature_train_rows'] and held_dates.sum()==plan['cohorts']['expected_mature_held_rows']
assert int(feature.loc[train_dates].isna().sum())==plan['cohorts']['expected_train_transition_unknown_rows']
assert feature.loc[held_dates].notna().all()
assert controls.loc[train_dates|held_dates].notna().all().all()
assert controls.loc[train_dates,'transition_unknown'].sum()==23 and controls.loc[held_dates,'transition_unknown'].sum()==0
augmented=base[base_names].copy()
for column in control_names:augmented[column]=controls[column]
augmented[name]=feature
meta=pd.DataFrame({'label_available_at':mature,'train_eligible':train_dates,'held_eligible':held_dates},index=frame.index)
static=pd.concat({'feature':augmented,'label':label.to_frame(),'meta':meta},axis=1)
static.index=pd.MultiIndex.from_arrays([frame.index,['SOXL_SOXX_CASH']*len(frame)],names=['datetime','instrument'])
dataset=DatasetH(handler=DataHandlerLP.from_df(static),segments={'train':tuple(plan['cohorts']['train_signal']),'held':tuple(plan['cohorts']['held_reused_diagnostic'])})
train_prepared=dataset.prepare('train',col_set=['feature','label','meta'])
held_prepared=dataset.prepare('held',col_set=['feature','label','meta'])
assert len(train_prepared)>=train_dates.sum() and len(held_prepared)>=held_dates.sum()
report={'status':'conditional_source_DatasetH_label_maturity_and_transition_preflight_passed_no_fit','plan_sha256':sha(card/'plan.json'),'mapped_source_sha256':spec['mapped_artifact_sha256'],'source_recorder_id':spec['recorder_id'],'parent_recorder_id':parent['recorder_id'],'original_primary_features':len(base_names),'common_control_coordinates':len(control_names),'candidate_coordinates':1,'training_mature_rows':int(train_dates.sum()),'held_mature_rows':int(held_dates.sum()),'training_last_label_available_at':str(mature.loc[train_dates].max()),'held_last_label_available_at':str(mature.loc[held_dates].max()),'train_segment_rows':len(train_prepared),'held_segment_rows':len(held_prepared),'candidate_train_unknown_rows':int(feature.loc[train_dates].isna().sum()),'candidate_held_unknown_rows':int(feature.loc[held_dates].isna().sum()),'train_transition_unknown_control_rows':int(controls.loc[train_dates,'transition_unknown'].sum()),'train_source_age_max_days':float(controls.loc[train_dates,'release_age_days'].max()),'held_source_age_max_days':float(controls.loc[held_dates,'release_age_days'].max()),'source_Recorder_loads_total_through_preflight':11,'DatasetH_builds_total_through_preflight':2,'new_model_fits':0,'new_native_accounts':0,'held_scores_computed':False,'historical_independence':False,'historical_first_public_bytes_qualified':False}
receipt=card/'preflight_result.json'
if receipt.exists():
 previous=json.loads(receipt.read_text());assert previous==report
else:atomic_json(report,receipt)
{k:report[k] for k in ('status','training_mature_rows','held_mature_rows','candidate_train_unknown_rows','candidate_held_unknown_rows')}


Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
Users of this version of Gym should be able to simply replace 'import gym' with 'import gymnasium as gym' in the vast majority of cases.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.


[40636:MainThread](2026-10-23 00:55:00,846) INFO - qlib.Initialization - [config.py:452] - default_conf: client.


[40636:MainThread](2026-10-23 00:55:00,848) INFO - qlib.Initialization - [__init__.py:75] - qlib successfully initialized based on client settings.


[40636:MainThread](2026-10-23 00:55:00,848) INFO - qlib.Initialization - [__init__.py:77] - data_path={'__DEFAULT_FREQ': PosixPath('/Users/rziiiii/dev/workspace/data/qlib/minute_information_pit_allocator_20261012/daily_prices')}


[40636:MainThread](2026-10-23 00:55:01,502) INFO - qlib.timer - [log.py:127] - Time cost: 0.002s | Loading data Done


[40636:MainThread](2026-10-23 00:55:01,502) INFO - qlib.timer - [log.py:127] - Time cost: 0.000s | fit & process data Done


[40636:MainThread](2026-10-23 00:55:01,503) INFO - qlib.timer - [log.py:127] - Time cost: 0.002s | Init data Done


{'status': 'conditional_source_DatasetH_label_maturity_and_transition_preflight_passed_no_fit',
 'training_mature_rows': 3014,
 'held_mature_rows': 940,
 'candidate_train_unknown_rows': 23,
 'candidate_held_unknown_rows': 0}

## 执行门

先运行无拟合 Qlib 预检；独立复核计划、预检代码与拟合代码哈希后，才允许两个固定 Ridge 拟合和一次配对区块检验。


In [2]:
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge
from sklearn.pipeline import make_pipeline
from src.analysis.seed_disagreement_information import full_calendar_bootstrap
from src.data.nas_market_data import archive_nas_recorder, _ssh
import shlex
review_path=card/'design_review_final.json';permit_path=card/'execution_approval.json'
if not review_path.exists() or not permit_path.exists():
 print('preflight_only: independent design review and exact execution permit absent')
else:
 review=json.loads(review_path.read_text());permit=json.loads(permit_path.read_text())
 notebook=json.loads((card/'stage1_ft900_export_unknown_information_20261023.ipynb').read_text())
 preflight_source=''.join(notebook['cells'][1]['source'])
 preflight_cell_hash=hashlib.sha256(preflight_source.encode()).hexdigest()
 fit_source=''.join(notebook['cells'][3]['source'])
 fit_hash=hashlib.sha256(fit_source.encode()).hexdigest()
 digest=sha(card/'plan.json');preflight_hash=sha(card/'preflight_result.json')
 assert review['verdict']=='APPROVE_DESIGN' and review['plan_sha256']==digest
 assert review['preflight_cell_source_sha256']==preflight_cell_hash
 assert review['fit_cell_source_sha256']==fit_hash
 assert permit['approved'] is True and permit['plan_sha256']==digest
 assert permit['review_sha256']==sha(review_path) and permit['fit_cell_source_sha256']==fit_hash
 assert permit['preflight_cell_source_sha256']==preflight_cell_hash
 assert permit['preflight_sha256']==preflight_hash
 assert report==json.loads((card/'preflight_result.json').read_text())
 state_path=card/'run_state.json';tracking=card/'qlib_recorder'
 state=json.loads(state_path.read_text()) if state_path.exists() else {'plan_sha256':digest,'fit_attempts':[],'fits_done':[],
   'artifact_sha256':{'models':{},'predictions':{}},'bootstrap_attempted':False,'new_native_accounts':0}
 assert state['plan_sha256']==digest
 if state.get('completed'):
  summary=json.loads((card/'summary.json').read_text())
  assert summary['plan_sha256']==digest and summary['nas_readback']['verified']
 elif state.get('analysis_done'):
  summary=json.loads((card/'summary.json').read_text())
 else:
  if state['bootstrap_attempted']:raise ValueError('blocked: fixed bootstrap attempt already consumed')
  tr=train_prepared.loc[train_prepared[('meta','train_eligible')].astype(bool)]
  he=held_prepared.loc[held_prepared[('meta','held_eligible')].astype(bool)]
  assert len(tr)==report['training_mature_rows'] and len(he)==report['held_mature_rows']
  yname='soxx_forward_five_open_log_return';name=plan['single_mechanism']['feature']
  ytr=tr[('label',yname)].to_numpy(float);yhe=he[('label',yname)].to_numpy(float)
  assert np.isfinite(ytr).all() and np.isfinite(yhe).all()
  inputs={'baseline':(tr['feature'][base_names+control_names].to_numpy(float),he['feature'][base_names+control_names].to_numpy(float)),
          'candidate':(tr['feature'][base_names+control_names+[name]].to_numpy(float),he['feature'][base_names+control_names+[name]].to_numpy(float))}
  assert int(np.isnan(inputs['candidate'][0][:,-1]).sum())==23 and np.isfinite(inputs['candidate'][1][:,-1]).all()
  models={};predictions={}
  start={'experiment_name':plan['experiment'],'uri':str(tracking)}
  if state.get('recorder_id'):start.update(recorder_id=state['recorder_id'],resume=True)
  with R.start(**start):
   recorder=R.get_recorder()
   if state.get('recorder_id'):
    assert state['recorder_id']==recorder.id and state['experiment_id']==str(recorder.experiment_id)
   state.update(recorder_id=recorder.id,experiment_id=str(recorder.experiment_id))
   atomic_json(state,state_path)
   artifact_root=tracking/state['experiment_id']/state['recorder_id']/'artifacts'
   if not state.get('sources_saved'):
    dataset.config(dump_all=True,recursive=True)
    recorder.save_objects(artifact_path='inputs',plan=plan,preflight=report,dataset=dataset,
                          mapped_signals=mapped)
    saved_dataset=recorder.load_object('inputs/dataset')
    assert len(saved_dataset.prepare('train',col_set=['feature','label','meta']))==report['train_segment_rows']
    assert len(saved_dataset.prepare('held',col_set=['feature','label','meta']))==report['held_segment_rows']
    state['sources_saved']=True;atomic_json(state,state_path)
   for family in ('baseline','candidate'):
    xtrain,xheld=inputs[family]
    if family in state['fits_done']:
     assert sha(artifact_root/'models'/family)==state['artifact_sha256']['models'][family]
     model=recorder.load_object('models/'+family)
    else:
     if family in state['fit_attempts'] or len(state['fit_attempts'])>=plan['budget']['model_fit_attempts_max']:
      raise ValueError('blocked: fixed model fit attempt consumed')
     state['fit_attempts'].append(family);atomic_json(state,state_path)
     model=make_pipeline(SimpleImputer(strategy='median'),StandardScaler(),Ridge(alpha=1.0))
     model.fit(xtrain,ytr)
     recorder.save_objects(artifact_path='models',**{family:model})
     state['artifact_sha256']['models'][family]=sha(artifact_root/'models'/family)
     state['fits_done'].append(family);atomic_json(state,state_path)
    if family in state.get('predictions_done',[]):
     assert sha(artifact_root/'predictions'/family)==state['artifact_sha256']['predictions'][family]
     saved=recorder.load_object('predictions/'+family)
     prediction=saved.prediction.to_numpy(float)
    else:
     if family in state.get('prediction_attempts',[]):raise ValueError('blocked: fixed prediction attempt consumed')
     state.setdefault('prediction_attempts',[]).append(family);atomic_json(state,state_path)
     prediction=model.predict(xheld)
     assert np.isfinite(prediction).all()
     saved=pd.DataFrame({'signal_date':he.index.get_level_values('datetime'),'prediction':prediction,'target':yhe})
     recorder.save_objects(artifact_path='predictions',**{family:saved})
     state['artifact_sha256']['predictions'][family]=sha(artifact_root/'predictions'/family)
     state.setdefault('predictions_done',[]).append(family);atomic_json(state,state_path)
    models[family],predictions[family]=model,prediction
    recorder.log_metrics(completed_fits=len(state['fits_done']),completed_predictions=len(state['predictions_done']))
   assert len(state['fits_done'])==2 and len(state['predictions_done'])==2
   dates=pd.DatetimeIndex(he.index.get_level_values('datetime'))
   daily=pd.DataFrame({'signal_date':dates,'target':yhe,'baseline_prediction':predictions['baseline'],
                       'candidate_prediction':predictions['candidate']})
   daily['baseline_squared_error']=(daily.target-daily.baseline_prediction)**2
   daily['candidate_squared_error']=(daily.target-daily.candidate_prediction)**2
   daily['paired_mse_improvement']=daily.baseline_squared_error-daily.candidate_squared_error
   assert daily.signal_date.is_monotonic_increasing and len(daily)==report['held_mature_rows']
   boundary=pd.Timestamp('2025-01-01')
   halves=[daily.loc[daily.signal_date.lt(boundary)],daily.loc[daily.signal_date.ge(boundary)]]
   assert min(map(len,halves))>300
   bootstrap=plan['information_gate']['paired_stationary_bootstrap']
   state['bootstrap_attempted']=True;atomic_json(state,state_path)
   interval,_,replicates=full_calendar_bootstrap(daily.paired_mse_improvement.to_numpy(float),
     mean_block=bootstrap['mean_block_sessions'],resamples=bootstrap['draws'],seed=bootstrap['seed'],
     alpha=bootstrap['one_sided_alpha'])
   coefficient=float(models['candidate'].named_steps['ridge'].coef_[-1])
   metrics={'baseline_held_mse':float(daily.baseline_squared_error.mean()),
            'candidate_held_mse':float(daily.candidate_squared_error.mean()),
            'held_paired_mse_improvement':float(daily.paired_mse_improvement.mean()),
            'held_half_paired_improvements':[float(x.paired_mse_improvement.mean()) for x in halves],
            'candidate_standardized_coordinate_coefficient':coefficient,
            'bootstrap_lower':interval['lower_bound'],'bootstrap_upper':interval['upper_bound'],
            'bootstrap':interval}
   gates={'coefficient_positive':coefficient>0,
          'full_held_mean_positive':metrics['held_paired_mse_improvement']>0,
          'both_halves_positive':all(x>0 for x in metrics['held_half_paired_improvements']),
          'bootstrap_lower_positive':metrics['bootstrap_lower']>0}
   passed=bool(all(gates.values()))
   daily['quarter']=daily.signal_date.dt.to_period('Q').astype(str)
   quarters=[{'quarter':q,'sessions':len(group),'paired_mean_mse_improvement':float(group.paired_mse_improvement.mean())}
             for q,group in daily.groupby('quarter',sort=True)]
   recorder.save_objects(artifact_path='analysis/daily',held=daily)
   recorder.save_objects(artifact_path='analysis',metrics=metrics,gates=gates,quarters=quarters,
                         bootstrap_replicates=replicates,information_gate_pass=passed)
   recorder.log_metrics(held_paired_mse_improvement=metrics['held_paired_mse_improvement'],
                        bootstrap_lower=metrics['bootstrap_lower'],information_gate_pass=int(passed))
  summary={'experiment':plan['experiment'],'plan_sha256':digest,
    'decision':'conditional_development_information_pass_source_PIT_still_required' if passed else
               'not_promoted_fixed_FT900_export_information_failed',
    'objective_achieved':False,'information_gate_pass':passed,
    'training_mature_rows':len(tr),'held_mature_rows':len(he),'metrics':metrics,'gates':gates,
    'quarterly_information_attribution':quarters,'new_model_fits':2,'new_bootstraps':1,'new_native_accounts':0,
    'reused_market_history':True,'historical_first_public_bytes_qualified':False,
    'independent_forward_qualified_days':0,'recorder_id':state['recorder_id'],
    'experiment_id':state['experiment_id'],'recorder_uri':str(tracking),
    'recorder_local_dir':str(tracking/state['experiment_id']/state['recorder_id']),
    'mapped_source_sha256':report['mapped_source_sha256'],
    'artifact_policy':plan['artifact_policy'],
    'limitations':['Original FT900 PDF first-public bytes are not historically certified; report date uses conservative next-Qlib-session availability',
                   'January 2012 definition transition has 23 intentionally unknown training feature rows',
                   'Information MSE is not a fee-net account result, nor independent forward evidence']}
  atomic_json(summary,card/'summary.json')
  atomic_json({'status':summary['decision'],'plan_sha256':digest,'recorder_id':state['recorder_id'],
    'metrics':metrics,'gates':gates,'new_model_fits':2,'new_native_accounts':0,
    'historical_first_public_bytes_qualified':False},card/'result_review.json')
  pd.DataFrame(quarters).to_csv(card/'quarterly_information_attribution.csv',index=False)
  artifacts=tracking/state['experiment_id']/state['recorder_id']/'artifacts'
  atomic_json({'recorder_id':state['recorder_id'],
    'entries':[{'path':str(x.relative_to(card)),'sha256':sha(x),'bytes':x.stat().st_size}
               for x in sorted(artifacts.rglob('*')) if x.is_file()]},card/'verification_manifest.json')
  state['analysis_done']=True;atomic_json(state,state_path)
 if state.get('analysis_done') and not state.get('completed'):
  from src.data.nas_market_data import _ssh
  manifest=json.loads((card/'verification_manifest.json').read_text())
  assert manifest['recorder_id']==summary['recorder_id']
  for entry in manifest['entries']:
   path=card/entry['path'];assert sha(path)==entry['sha256'] and path.stat().st_size==entry['bytes']
  binding_path=card/'nas_binding.json'
  if not binding_path.exists():
   binding=json.loads((root/'research/stage1_rolling126_shortfall_objective_20261022/nas_binding.json').read_text())
   assert binding.get('verified') is True and binding.get('authority')=='yanxuan_nas'
   binding.pop('evidence',None);binding['research_card']=str(card.relative_to(root));atomic_json(binding,binding_path)
  if 'nas_evidence' not in summary:
   summary['nas_evidence']=archive_nas_recorder(binding_path,workspace_root=root,card=card,
      recorder_id=summary['recorder_id'],experiment_id=summary['experiment_id'])
  binding=json.loads(binding_path.read_text());evidence=summary['nas_evidence']
  assert binding['evidence']['sha256']==evidence['sha256']
  remote_sha=_ssh(binding['ssh'],'sha256sum '+shlex.quote(evidence['nas_path'])).split()[0]
  assert remote_sha==evidence['sha256']
  summary['nas_readback']={'sha256':remote_sha,'bytes':evidence['size_bytes'],'verified':True}
  atomic_json(summary,card/'summary.json')
  state['completed']=True;atomic_json(state,state_path)
 print({k:summary[k] for k in ('decision','information_gate_pass','new_model_fits','new_native_accounts','recorder_id')})


[40636:MainThread](2026-10-23 00:55:01,524) WARNING - qlib.workflow - [expm.py:231] - No valid experiment found. Create a new experiment with name stage1_ft900_export_unknown_information_20261023.


[40636:MainThread](2026-10-23 00:55:01,525) INFO - qlib.workflow - [exp.py:258] - Experiment 750328884082775760 starts running ...


[40636:MainThread](2026-10-23 00:55:01,606) INFO - qlib.workflow - [recorder.py:345] - Recorder 541223b7ae38472bbba6e71eff786cf3 starts running under Experiment 750328884082775760 ...


[40636:MainThread](2026-10-23 00:55:04,849) INFO - qlib.timer - [log.py:127] - Time cost: 0.001s | waiting `async_log` Done


{'decision': 'not_promoted_fixed_FT900_export_information_failed', 'information_gate_pass': False, 'new_model_fits': 2, 'new_native_accounts': 0, 'recorder_id': '541223b7ae38472bbba6e71eff786cf3'}


## 中文结论

本次运行 Qlib Recorder `541223b7ae38472bbba6e71eff786cf3`，3014 条成熟训练与 940 条复用诊断；来源 2012 定义断点的 23 条训练特征值保持未知。两次匹配 Ridge 拟合、一次 63 日均块／10000 次／单侧 alpha=.025/24 的配对检验，零经济账户。对照/候选 MSE 为 0.002524722/0.002524778；新增系数 0.0014819 为正，但全段改善 -5.5478e-08、第二半改善 -5.0498e-06、校正下界 -6.7369e-05 均未过门。2025 四季度全负，2024Q3、2026Q2/Q3 的局部正值不能转成后验开关。固定版 `not_promoted`，不进入 0bp 账户。来源首次公开字节与独立前向仍未认证；下一步寻找实质不同的 Stage1 决策时点机制。证据见 `summary.json`、`failure_attribution.json`、`readback_verification.json`、`conclusion.md`，NAS SHA `cb5b55c11d553e073a557a094da88fded4e1d7055e0c1efc9a9e05b752afb888`。
